# Chest X-ray14 (112k): fine-tuning completo de los 4 modelos

**Autor:** Alfredo Gallegos Lopez, Universidad Panamericana, Aguascalientes. Proyecto final de Deep Learning (extensión).

Mismo protocolo que `ChestXray14_112k_bajo_consumo.ipynb` (dataset completo, split por paciente, 4 modelos + ensemble, mismas métricas y modo de bajo consumo), con **una sola diferencia de diseño**: el **fine-tuning es completo**. Se entrenan **todas** las capas de cada modelo, no solo el último bloque y la capa de salida.

- **Tasa de aprendizaje:** 3e-5, en lugar de 1e-4. Al ajustar toda la red, una tasa alta destruiría las representaciones aprendidas en ImageNet; 3e-5 es un valor habitual para fine-tuning completo con Adam, tanto en CNN como en Transformers.
- **Objetivo:** responder el trabajo futuro (i) del paper. ¿Cuánto mejora cada arquitectura al adaptar toda la red? ¿Los modelos con atención (ViT, MaxViT), de los que antes solo se ajustaba el 8% y el 30% de sus parámetros, alcanzan o superan a las CNN?
- **Salidas separadas:** `results/full_112k_finetuning_completo/` y `checkpoints/112k_ft_completo/`.

Para correrlo: `python run_notebook.py ChestXray14_112k_finetuning_completo.ipynb <kernel>` o la cola `python cola_extensiones.py`. Es reanudable desde los checkpoints.

## 1. Setup

In [ ]:
import os
import copy
import json
import time
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.utils.data as data
from torchvision import models, transforms
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PROJECT_ROOT = Path(".").resolve()
ARCHIVE_PATH = PROJECT_ROOT / "archive.zip"
CACHE_SIZE = 256  # las radiografias (1024x1024) se guardan redimensionadas a este tamano; el modelo luego usa IMG_SIZE
IMAGES_DIR = PROJECT_ROOT / f"images_{CACHE_SIZE}"
RESULTS_DIR = PROJECT_ROOT / "results" / "full_112k_finetuning_completo"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "112k_ft_completo"  # checkpoint por epoca: si la corrida se corta, al volver a correr continua
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

assert ARCHIVE_PATH.exists(), (
    f"No se encontro {ARCHIVE_PATH}. Coloca archive.zip (dataset de Kaggle nih-chest-xrays/data) "
    "en la raiz del proyecto antes de continuar."
)

# ---- Configuracion del experimento (ajustable) ----
N_SAMPLES = 112120         # dataset completo
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 3e-5  # fine-tuning completo: tasa menor que con fine-tuning parcial (1e-4)
FULL_FINETUNING = True  # todas las capas entrenables
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # el resto (0.15) es test; el split se hace por paciente
N_BOOTSTRAP = 1000        # remuestreos del test set para intervalos de confianza al 95%

# ---- Modo de bajo consumo (no cambia el protocolo ni los resultados, solo el ritmo) ----
GPU_DUTY_SLEEP = 1.0              # pausa = este factor x tiempo de computo de cada lote (1.0: GPU activa ~50% del tiempo)
COOLDOWN_BETWEEN_EPOCHS_SEC = 60  # enfriamiento entre epocas
COOLDOWN_BETWEEN_MODELS_SEC = 180 # enfriamiento entre modelos

DISEASE_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(DISEASE_LABELS)
print(f"{NUM_CLASSES} clases:", DISEASE_LABELS)


## 2. Datos

Se usa el dataset completo. Cada radiografia se lee de `archive.zip` y se guarda **una sola vez** redimensionada a `CACHE_SIZE` px en `images_256/` (si la cache ya existe, este paso se salta). Leer PNGs de 1024x1024 en cada epoca haria que la carga de datos dominara el tiempo; como los modelos usan 224x224, redimensionar antes no pierde informacion que el modelo fuera a usar.

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    names = zf.namelist()
    print(f"Total de entradas en el zip: {len(names)}")

    # Encontrar el CSV de etiquetas (normalmente 'Data_Entry_2017.csv')
    csv_candidates = [n for n in names if n.lower().endswith(".csv") and "data_entry" in n.lower()]
    if not csv_candidates:
        csv_candidates = [n for n in names if n.lower().endswith(".csv")]
    assert csv_candidates, "No se encontro ningun CSV de etiquetas dentro de archive.zip. Revisa el contenido del zip manualmente."
    labels_csv_name = csv_candidates[0]
    print("CSV de etiquetas encontrado:", labels_csv_name)

    with zf.open(labels_csv_name) as f:
        df_labels = pd.read_csv(f)

print(df_labels.shape)
df_labels.head()


In [ ]:
# Columna con el nombre de archivo de imagen (varia ligeramente segun la version del CSV)
image_col_candidates = [c for c in df_labels.columns if "image" in c.lower() and "index" in c.lower()]
IMAGE_COL = image_col_candidates[0] if image_col_candidates else df_labels.columns[0]
LABEL_COL = [c for c in df_labels.columns if "finding" in c.lower() and "label" in c.lower()][0]
print("Columna de imagen:", IMAGE_COL, "| Columna de etiquetas:", LABEL_COL)

# Muestreo aleatorio de N_SAMPLES filas (con semilla fija para reproducibilidad)
df_sample = df_labels.sample(n=min(N_SAMPLES, len(df_labels)), random_state=SEED).reset_index(drop=True)
print(f"Subconjunto muestreado: {len(df_sample)} imagenes")

# Vector binario de 14 clases a partir de 'Finding Labels' (pipe-separated; 'No Finding' = todo ceros)
def labels_to_vector(finding_labels_str):
    present = set(finding_labels_str.split("|"))
    return np.array([1.0 if label in present else 0.0 for label in DISEASE_LABELS], dtype=np.float32)

df_sample["label_vector"] = df_sample[LABEL_COL].apply(labels_to_vector)
print("Ejemplo:", df_sample[IMAGE_COL].iloc[0], "->", df_sample[LABEL_COL].iloc[0], "->", df_sample["label_vector"].iloc[0])


In [ ]:
# Cache de imagenes redimensionadas: se leen del zip y se guardan a CACHE_SIZE px, en paralelo.
# Las funciones viven en chestxray_data.py porque los procesos del Pool (en Windows) tienen que poder importarlas.
from multiprocessing import Pool
from chestxray_data import init_zip_worker, resize_from_zip

wanted_filenames = set(df_sample[IMAGE_COL])
with zipfile.ZipFile(ARCHIVE_PATH, "r") as zf:
    name_by_basename = {Path(n).name: n for n in zf.namelist() if Path(n).name in wanted_filenames}
missing = sorted(wanted_filenames - set(name_by_basename))

tasks = [(name_by_basename[f], str(IMAGES_DIR / f), CACHE_SIZE)
         for f in sorted(wanted_filenames) if f in name_by_basename and not (IMAGES_DIR / f).exists()]
n_cached = len(wanted_filenames) - len(missing) - len(tasks)
print(f"Ya en cache: {n_cached} | Por procesar: {len(tasks)} | No encontradas en el zip: {len(missing)}")

if tasks:
    n_proc = 4  # pocos procesos: menor consumo del equipo
    t0 = time.time()
    with Pool(n_proc, initializer=init_zip_worker, initargs=(str(ARCHIVE_PATH),)) as pool:
        for k, _ in enumerate(pool.imap_unordered(resize_from_zip, tasks, chunksize=64), start=1):
            if k % 10000 == 0 or k == len(tasks):
                print(f"  {k}/{len(tasks)} imagenes procesadas ({time.time() - t0:.0f}s)")
    print(f"Cache lista en {time.time() - t0:.0f}s con {n_proc} procesos")

if missing:
    print("Ejemplos de imagenes no encontradas en el zip (revisar nombres):", missing[:5])
    df_sample = df_sample[~df_sample[IMAGE_COL].isin(missing)].reset_index(drop=True)
    print(f"Continuando con {len(df_sample)} imagenes validas.")

In [ ]:
# La clase del Dataset vive en chestxray_data.py (junto a este notebook): en Windows los workers del
# DataLoader se lanzan con 'spawn' y no pueden importar una clase definida dentro del notebook.
from chestxray_data import ChestXray14Dataset

### 2.1 Preprocesamiento, split por paciente y DataLoaders

Esta celda hace cuatro cosas:
1. **Transforms**: redimensiona a 224x224 y normaliza con la media y desviacion de ImageNet (lo que esperan los modelos preentrenados). En entrenamiento agrega volteo horizontal como *data augmentation*.
2. **Split por paciente** (70/15/15 % de las imagenes): un mismo paciente tiene varias radiografias; si quedaran repartidas entre entrenamiento y prueba, el modelo se evaluaria sobre pacientes ya vistos. Se barajan los pacientes con semilla fija y se asignan en orden hasta llenar cada conjunto. Un `assert` verifica que ningun paciente quede en dos conjuntos.
3. **Positivos por clase** en cada conjunto (`prevalence_df`): muestra el desbalance (Hernia es muy rara).
4. **DataLoaders** con lotes de 32. La clase `ChestXray14Dataset` (en `chestxray_data.py`) lee cada PNG, lo convierte a 3 canales y le aplica el transform.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Split por PACIENTE, no por imagen: en ChestX-ray14 un mismo paciente tiene varias radiografias, y si quedan
# repartidas entre train y test el modelo se evalua sobre pacientes que ya vio (fuga de datos que infla las metricas).
# Se barajan los pacientes y se asignan en orden hasta acumular ~70% / 15% / 15% de las imagenes.
PATIENT_COL = "Patient ID"
patients = df_sample[PATIENT_COL].unique()
np.random.default_rng(SEED).shuffle(patients)
imgs_per_patient = df_sample[PATIENT_COL].value_counts().reindex(patients).to_numpy()
cum_frac = np.cumsum(imgs_per_patient) / len(df_sample)
patient_split = dict(zip(patients, np.where(cum_frac <= TRAIN_FRAC, "train",
                                            np.where(cum_frac <= TRAIN_FRAC + VAL_FRAC, "val", "test"))))
split_col = df_sample[PATIENT_COL].map(patient_split)
df_train = df_sample[split_col == "train"].reset_index(drop=True)
df_val = df_sample[split_col == "val"].reset_index(drop=True)
df_test = df_sample[split_col == "test"].reset_index(drop=True)

split_patients = {s: set(d[PATIENT_COL]) for s, d in [("train", df_train), ("val", df_val), ("test", df_test)]}
assert not (split_patients["train"] & split_patients["val"]) and not (split_patients["train"] & split_patients["test"]) \
    and not (split_patients["val"] & split_patients["test"]), "Hay pacientes compartidos entre splits"
for s, d in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    print(f"{s}: {len(d)} imagenes / {d[PATIENT_COL].nunique()} pacientes")
print("Pacientes compartidos entre splits: 0")

# Positivos por clase en cada split (Hernia es muy rara: sus metricas por clase son poco estables)
prevalence_df = pd.DataFrame({
    s: np.stack(d["label_vector"]).sum(axis=0).astype(int) for s, d in [("train", df_train), ("val", df_val), ("test", df_test)]
}, index=DISEASE_LABELS)
prevalence_df["prevalencia_total_%"] = (100 * prevalence_df[["train", "val", "test"]].sum(axis=1) / len(df_sample)).round(2)
print(prevalence_df)

train_dataset = ChestXray14Dataset(df_train, IMAGES_DIR, IMAGE_COL, train_transform)
val_dataset = ChestXray14Dataset(df_val, IMAGES_DIR, IMAGE_COL, eval_transform)
test_dataset = ChestXray14Dataset(df_test, IMAGES_DIR, IMAGE_COL, eval_transform)

# persistent_workers: evita relanzar los procesos en cada epoca (costoso en Windows, donde se usa 'spawn')
NUM_WORKERS = 2  # bajo consumo: menos procesos de lectura
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0)
train_loader = data.DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = data.DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = data.DataLoader(test_dataset, shuffle=False, **loader_kwargs)


In [ ]:
# Vistazo rapido a un batch, para confirmar que el pipeline de datos esta correcto antes de entrenar nada
imgs, labels = next(iter(train_loader))
print("Batch de imagenes:", imgs.shape, "| Batch de etiquetas:", labels.shape)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    present = [DISEASE_LABELS[j] for j, v in enumerate(labels[i]) if v == 1]
    ax.imshow(img)
    ax.set_title(", ".join(present) if present else "No Finding", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_batch.png", dpi=120)
plt.show()


## 3. Modelos

Cuatro modelos preentrenados en ImageNet (`torchvision.models`):

- **DenseNet121** y **ResNet18**: CNN.
- **ViT-B/16**: Transformer puro.
- **MaxViT-T**: hibrido CNN + Transformer (bloques convolucionales MBConv combinados con auto-atencion local por bloques y global por rejilla en cada etapa).

En todos el backbone queda congelado salvo el ultimo bloque, y se reemplaza la capa final por una de 14 salidas (clasificacion multi-etiqueta, sin softmax; se usa `BCEWithLogitsLoss` mas adelante). En MaxViT-T se descongela solo la ultima capa de la ultima etapa (la etapa completa tiene 17.5M parametros, mas del doble que lo que entrenan los otros modelos), para mantener una capacidad entrenable comparable.

Ademas se evalua un **ensemble** (promedio de las probabilidades de los 4 modelos), que no requiere entrenamiento adicional.

**En este notebook:** con `FULL_FINETUNING = True` se descongelan **todas** las capas (fine-tuning completo).

In [ ]:
def freeze_all(module):
    for p in module.parameters():
        p.requires_grad = False


def build_model(name):
    """Modelo preentrenado en ImageNet con nueva capa de 14 salidas y fine-tuning parcial (solo ultimo bloque + salida entrenables)."""
    if name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, NUM_CLASSES)
        for p in model.features.denseblock4.parameters():
            p.requires_grad = True
        for p in model.features.norm5.parameters():
            p.requires_grad = True

    elif name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        for p in model.layer4.parameters():
            p.requires_grad = True

    elif name == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, NUM_CLASSES)
        for p in model.encoder.layers[-1].parameters():
            p.requires_grad = True

    elif name == "maxvit_t":
        # Hibrido CNN + Transformer. Se descongela la ultima capa MaxViT de la ultima etapa y la cabeza de clasificacion
        model = models.maxvit_t(weights=models.MaxVit_T_Weights.IMAGENET1K_V1)
        freeze_all(model)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
        for p in model.blocks[-1].layers[-1].parameters():
            p.requires_grad = True
        for p in model.classifier.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Modelo desconocido: {name}")

    if FULL_FINETUNING:  # fine-tuning completo: se descongelan todas las capas del modelo preentrenado
        for p in model.parameters():
            p.requires_grad = True
    return model.to(device)


def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


MODEL_NAMES = ["densenet121", "resnet18", "vit_b_16", "maxvit_t"]
for name in MODEL_NAMES:
    m = build_model(name)
    total, trainable = count_trainable_params(m)
    print(f"{name}: {total:,} parametros totales, {trainable:,} entrenables")
    del m


## 4. Entrenamiento

Mismo protocolo experimental para los 4 modelos: mismo split, misma resolucion, mismo numero de epocas, mismo optimizador. El loop esta adaptado del patron `run_epoch`/`fit` visto en `CNNs_Fundamentals_PyTorch.ipynb`, cambiando a `BCEWithLogitsLoss` para clasificacion multi-etiqueta.

- **Fine-tuning parcial:** se parte de los pesos de ImageNet y se ajustan solo el ultimo bloque y la capa de salida (el resto queda congelado): 31% / 75% / 8% / 30% de los parametros en DenseNet121 / ResNet18 / ViT-B/16 / MaxViT-T. Razones: las primeras capas aprenden rasgos genericos que se transfieren bien; con 8,400 imagenes ajustar toda la red (hasta 86M parametros en ViT) arriesga sobreajuste; el presupuesto de computo no permite fine-tuning completo de 4 modelos; y la misma regla para todos mantiene la comparacion justa.
- **Mejor epoca:** al final del entrenamiento se restauran los pesos de la epoca con menor val loss (early stopping "a posteriori"); asi ningun modelo se evalua ya sobreajustado.
- **Reanudable:** al final de cada epoca se guarda un checkpoint en `checkpoints/12k/<modelo>.pt` (pesos, optimizador, historial, mejor epoca). Si la ejecucion se corta, al volver a correr el notebook cada modelo continua desde su ultima epoca guardada, y los modelos ya terminados solo se cargan. Para entrenar desde cero hay que borrar `checkpoints/12k/`.
- **Costo medido de forma justa:** antes de medir se hace un paso de calentamiento (seleccion inicial de kernels en la GPU y arranque de los workers del DataLoader), y cada modelo entrenado se pasa a CPU para que la memoria pico del siguiente sea solo la suya.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Una pasada por el loader (entrena si recibe optimizer); con pausas de bajo consumo. Regresa (perdida promedio, segundos en pausa)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, n_batches, slept = 0.0, 0, 0.0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            t_batch = time.time()
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs)
            loss = criterion(preds, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()  # .item() espera a que la GPU termine: t_batch mide computo real
            n_batches += 1
            if GPU_DUTY_SLEEP > 0:  # bajo consumo: la GPU descansa proporcionalmente a lo que trabajo
                pause = GPU_DUTY_SLEEP * (time.time() - t_batch)
                time.sleep(pause)
                slept += pause
    return total_loss / max(n_batches, 1), slept


def warmup(model, train_loader, val_loader, criterion):
    """Un forward/backward de entrenamiento y un forward de validacion, sin actualizar pesos, antes de medir tiempos.
    Absorbe la seleccion inicial de kernels de la GPU y el arranque de los workers del DataLoader, que si no
    se cargarian solo a la primera epoca del primer modelo."""
    state = copy.deepcopy(model.state_dict())  # en modo train BatchNorm actualiza sus estadisticas: se restauran despues
    model.train()
    imgs, labels = next(iter(train_loader))
    criterion(model(imgs.to(device)), labels.to(device)).backward()
    model.zero_grad(set_to_none=True)
    model.eval()
    with torch.no_grad():
        imgs, _ = next(iter(val_loader))
        model(imgs.to(device))
    model.load_state_dict(state)
    if device.type == "cuda":
        torch.cuda.synchronize()


def save_checkpoint(path, ckpt):
    """Escritura atomica: si el equipo se apaga a mitad de la escritura, el checkpoint anterior queda intacto."""
    tmp = path.with_suffix(".tmp")
    torch.save(ckpt, tmp)
    os.replace(tmp, path)


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE, checkpoint_path=None):
    """Entrena con BCE + Adam (reanudando del checkpoint si existe), restaura la mejor epoca y regresa (historial, memoria pico MB, mejor epoca)."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    history = {"train_loss": [], "val_loss": [], "epoch_time_sec": []}
    best_val_loss, best_epoch, best_state = float("inf"), -1, None
    start_epoch, prev_peak_mb = 0, 0.0
    if checkpoint_path is not None and checkpoint_path.exists():
        # En CPU: load_state_dict copia los pesos a la GPU; si se cargara directo en la GPU, la copia de los mejores
        # pesos (best_state) tambien ocuparia memoria de la GPU e inflaria la memoria pico medida al reanudar.
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        history, best_val_loss, best_epoch, best_state = ckpt["history"], ckpt["best_val_loss"], ckpt["best_epoch"], ckpt["best_state"]
        start_epoch, prev_peak_mb = ckpt["epochs_done"], ckpt["peak_memory_mb"]
        print(f"  checkpoint encontrado: {start_epoch}/{epochs} epocas ya entrenadas, se continua desde ahi")

    if start_epoch < epochs:
        warmup(model, train_loader, val_loader, criterion)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        train_loss, slept_train = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, slept_val = run_epoch(model, val_loader, criterion, optimizer=None)
        epoch_time = time.time() - t0 - slept_train - slept_val  # solo computo activo (sin pausas de bajo consumo)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["epoch_time_sec"].append(epoch_time)
        if val_loss < best_val_loss:
            best_val_loss, best_epoch = val_loss, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = "  <- mejor hasta ahora" if best_epoch == epoch else ""
        print(f"  epoch {epoch+1}/{epochs} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - {epoch_time:.1f}s{marker}")

        peak_memory_mb = max(prev_peak_mb, torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else 0.0)
        if checkpoint_path is not None:
            save_checkpoint(checkpoint_path, {
                "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(), "history": history,
                "best_val_loss": best_val_loss, "best_epoch": best_epoch, "best_state": best_state,
                "epochs_done": epoch + 1, "peak_memory_mb": peak_memory_mb,
            })
        if epoch + 1 < epochs and COOLDOWN_BETWEEN_EPOCHS_SEC > 0:
            time.sleep(COOLDOWN_BETWEEN_EPOCHS_SEC)  # enfriamiento (fuera de la medicion de tiempo)

    peak_memory_mb = max(prev_peak_mb, torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device.type == "cuda" else 0.0)
    model.load_state_dict(best_state)
    print(f"  -> se restauran los pesos de la epoca {best_epoch+1} (val_loss {best_val_loss:.4f})")
    return history, peak_memory_mb, best_epoch + 1


### 4.1 Entrenamiento de los 4 modelos

Para cada modelo: se construye con `build_model`, se entrena con `fit` (que reanuda desde su checkpoint si existe) y al terminar se pasa a CPU, para que la memoria pico del siguiente modelo no incluya a este. Se guardan parametros, historial de perdidas, mejor epoca, memoria pico y tiempo por epoca.

In [ ]:
trained_models = {}
training_results = {}

for name in MODEL_NAMES:
    print(f"\n=== Entrenando {name} ===")
    if device.type == "cuda":
        torch.cuda.empty_cache()
    model = build_model(name)
    total_params, trainable_params = count_trainable_params(model)
    history, peak_memory_mb, best_epoch = fit(model, train_loader, val_loader, checkpoint_path=CHECKPOINT_DIR / f"{name}.pt")

    # Se pasa a CPU para liberar la GPU: la memoria pico del siguiente modelo no incluye a este
    trained_models[name] = model.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if name != MODEL_NAMES[-1] and COOLDOWN_BETWEEN_MODELS_SEC > 0:
        print(f"  enfriamiento de {COOLDOWN_BETWEEN_MODELS_SEC}s antes del siguiente modelo")
        time.sleep(COOLDOWN_BETWEEN_MODELS_SEC)
    training_results[name] = {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "history": history,
        "best_epoch": best_epoch,
        "peak_memory_mb": peak_memory_mb,
        "avg_epoch_time_sec": float(np.mean(history["epoch_time_sec"])),
    }

In [ ]:
MODEL_COLORS = {"densenet121": "#2a78d6", "resnet18": "#eb6834", "vit_b_16": "#1baf7a", "maxvit_t": "#eda100", "ensemble": "#6250d6"}  # mismo color por modelo en todas las figuras
MODEL_MARKERS = {"densenet121": "o", "resnet18": "s", "vit_b_16": "^", "maxvit_t": "D", "ensemble": "*"}  # segunda codificacion, ademas del color


def style_axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color("#999999")
    ax.spines["bottom"].set_color("#999999")
    ax.tick_params(colors="#444444")
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    ax.set_axisbelow(True)


fig, ax = plt.subplots(figsize=(7, 5))
epochs_axis = np.arange(1, EPOCHS + 1)
for name in MODEL_NAMES:
    val_losses = training_results[name]["history"]["val_loss"]
    best = training_results[name]["best_epoch"]
    ax.plot(epochs_axis, val_losses, marker="o", markersize=6, linewidth=2, color=MODEL_COLORS[name], label=name)
    ax.plot(best, val_losses[best - 1], marker="*", markersize=16, color=MODEL_COLORS[name],
            markeredgecolor="white", markeredgewidth=1.2, linestyle="none")
ax.plot([], [], marker="*", markersize=12, color="#666666", linestyle="none", label="mejor epoca (pesos usados)")
ax.set_xticks(epochs_axis)
ax.set_xlabel("Epoca")
ax.set_ylabel("Val loss (BCE)")
ax.set_title("Curva de convergencia por modelo")
style_axes(ax)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "val_loss_curves.png", dpi=150)
plt.show()

## 5. Evaluacion

Sobre el test set (pacientes que ningun modelo vio), para cada modelo:

- **AUROC** por clase y su promedio (macro): metrica principal, estandar en ChestX-ray14/CheXNet y no depende de ningun umbral.
- **F1 macro/micro con umbral por clase**: el umbral de cada clase se elige maximizando F1 en **validacion** y se aplica sin cambios al test. Con prevalencias de ~1-15%, un umbral fijo de 0.5 casi nunca predice positivos; se reporta tambien F1 @0.5 como referencia.
- **Accuracy por etiqueta**: se reporta solo como contexto, junto a la de un clasificador trivial que predice siempre "sin hallazgo" (con clases tan desbalanceadas, la accuracy es casi la misma para cualquier modelo).
- **Ensemble**: promedio de las probabilidades de los 4 modelos, con su propio umbral por clase elegido en validacion.
- **Intervalos de confianza al 95%** por bootstrap del test set (`N_BOOTSTRAP` remuestreos), y comparacion **pareada** entre modelos (mismos remuestreos para todos): una diferencia es significativa si su intervalo no incluye 0.
- Las probabilidades de validacion y prueba se guardan en `results/predictions.npz`, para recalcular metricas sin volver a entrenar.

In [ ]:
def predict(model, loader):
    """Probabilidades (sigmoide) y etiquetas reales de todo el loader; deja el modelo en CPU al terminar."""
    model.to(device).eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            t_batch = time.time()
            all_probs.append(torch.sigmoid(model(imgs.to(device))).cpu().numpy())  # .cpu() espera a la GPU
            all_labels.append(labels.numpy())
            if GPU_DUTY_SLEEP > 0:
                time.sleep(GPU_DUTY_SLEEP * (time.time() - t_batch))
    model.cpu()
    return np.concatenate(all_probs), np.concatenate(all_labels)


def auroc_per_class(y_true, y_prob):
    """AUROC por clase; NaN si la clase no tiene positivos (o negativos) en y_true."""
    out = np.full(NUM_CLASSES, np.nan)
    for j in range(NUM_CLASSES):
        if 0 < y_true[:, j].sum() < len(y_true):
            out[j] = roc_auc_score(y_true[:, j], y_prob[:, j])
    return out


def tune_thresholds(y_true, y_prob, grid=np.linspace(0.01, 0.99, 99)):
    """Umbral por clase que maximiza F1 en validacion (0.5 si la clase no tiene positivos en validacion)."""
    thresholds = np.full(NUM_CLASSES, 0.5)
    for j in range(NUM_CLASSES):
        if y_true[:, j].sum() == 0:
            continue
        f1s = [f1_score(y_true[:, j], y_prob[:, j] >= t, zero_division=0) for t in grid]
        thresholds[j] = grid[int(np.argmax(f1s))]
    return thresholds


def summary_metrics(y_true, y_prob, thresholds):
    """AUROC macro, F1 macro/micro con los umbrales por clase y con 0.5, y exactitud por etiqueta con 0.5."""
    y_pred_tuned = (y_prob >= thresholds).astype(np.float32)
    y_pred_05 = (y_prob >= 0.5).astype(np.float32)
    return {
        "auroc_macro": float(np.nanmean(auroc_per_class(y_true, y_prob))),
        "f1_macro_tuned": float(f1_score(y_true, y_pred_tuned, average="macro", zero_division=0)),
        "f1_micro_tuned": float(f1_score(y_true, y_pred_tuned, average="micro", zero_division=0)),
        "f1_macro_at_0.5": float(f1_score(y_true, y_pred_05, average="macro", zero_division=0)),
        "f1_micro_at_0.5": float(f1_score(y_true, y_pred_05, average="micro", zero_division=0)),
        "accuracy_at_0.5": float((y_pred_05 == y_true).mean()),
    }


BOOT_KEYS = ["auroc_macro", "f1_macro_tuned", "f1_micro_tuned"]

def boot_metrics(y_true, y_prob, thresholds):
    """Solo las metricas con intervalo de confianza (mas rapido con ~17k imagenes de prueba)."""
    y_pred = (y_prob >= thresholds).astype(np.float32)
    return {
        "auroc_macro": float(np.nanmean(auroc_per_class(y_true, y_prob))),
        "f1_macro_tuned": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_micro_tuned": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
    }

def bootstrap(y_true, probs, thresholds, n=N_BOOTSTRAP):
    """Remuestreo del test set con los MISMOS indices para todos los modelos (permite comparar en pareado)."""
    rng = np.random.default_rng(SEED)
    samples = {name: {k: [] for k in BOOT_KEYS} for name in probs}
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        for name in probs:
            m = boot_metrics(y_true[idx], probs[name][idx], thresholds[name])
            for k in BOOT_KEYS:
                samples[name][k].append(m[k])
    return {name: {k: np.array(v) for k, v in d.items()} for name, d in samples.items()}


def ci95(values):
    return [float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))]


val_probs, test_probs, thresholds = {}, {}, {}
for name in MODEL_NAMES:
    val_probs[name], y_val = predict(trained_models[name], val_loader)
    test_probs[name], y_test = predict(trained_models[name], test_loader)
    thresholds[name] = tune_thresholds(y_val, val_probs[name])

# Ensemble: promedio de las probabilidades de los modelos (sin entrenamiento adicional), con su propio umbral por clase
val_probs["ensemble"] = np.mean([val_probs[n] for n in MODEL_NAMES], axis=0)
test_probs["ensemble"] = np.mean([test_probs[n] for n in MODEL_NAMES], axis=0)
thresholds["ensemble"] = tune_thresholds(y_val, val_probs["ensemble"])
EVAL_NAMES = MODEL_NAMES + ["ensemble"]

np.savez_compressed(RESULTS_DIR / "predictions.npz", y_val=y_val, y_test=y_test,
                    **{f"val_{n}": val_probs[n] for n in EVAL_NAMES}, **{f"test_{n}": test_probs[n] for n in EVAL_NAMES})

eval_results = {}
for name in EVAL_NAMES:
    eval_results[name] = summary_metrics(y_test, test_probs[name], thresholds[name])
    eval_results[name]["auroc_per_class"] = dict(zip(DISEASE_LABELS, auroc_per_class(y_test, test_probs[name]).tolist()))
    eval_results[name]["thresholds"] = dict(zip(DISEASE_LABELS, thresholds[name].tolist()))

trivial_accuracy = float((y_test == 0).mean())
print(f"Accuracy de referencia (predecir siempre 'sin hallazgo'): {trivial_accuracy:.4f}\n")

t0 = time.time()
boot = bootstrap(y_test, test_probs, thresholds)
print(f"Bootstrap ({N_BOOTSTRAP} remuestreos): {time.time() - t0:.0f}s")
for name in EVAL_NAMES:
    eval_results[name]["ci95"] = {k: ci95(boot[name][k]) for k in BOOT_KEYS}

pairwise = {}
for i, a in enumerate(EVAL_NAMES):
    for b in EVAL_NAMES[i + 1:]:
        for k in BOOT_KEYS:
            diff = boot[a][k] - boot[b][k]
            lo, hi = ci95(diff)
            pairwise[f"{a} - {b} | {k}"] = {
                "diff": eval_results[a][k] - eval_results[b][k], "ci95": [lo, hi], "significativa": bool(lo > 0 or hi < 0),
            }

def fmt_ci(name, k):
    lo, hi = eval_results[name]["ci95"][k]
    return f"{eval_results[name][k]:.3f} [{lo:.3f}, {hi:.3f}]"

comparison_df = pd.DataFrame({
    "Modelo": EVAL_NAMES,
    "AUROC macro [IC95]": [fmt_ci(n, "auroc_macro") for n in EVAL_NAMES],
    "F1 macro umbral [IC95]": [fmt_ci(n, "f1_macro_tuned") for n in EVAL_NAMES],
    "F1 micro umbral [IC95]": [fmt_ci(n, "f1_micro_tuned") for n in EVAL_NAMES],
    "F1 macro @0.5": [round(eval_results[n]["f1_macro_at_0.5"], 4) for n in EVAL_NAMES],
    "Accuracy @0.5": [round(eval_results[n]["accuracy_at_0.5"], 4) for n in EVAL_NAMES],
    "Mejor epoca": [training_results[n]["best_epoch"] if n in training_results else "-" for n in EVAL_NAMES],
}).set_index("Modelo")
display(comparison_df)

pairwise_df = pd.DataFrame([
    {"Comparacion": key.split(" | ")[0], "Metrica": key.split(" | ")[1], "Diferencia": round(v["diff"], 4),
     "IC95": f"[{v['ci95'][0]:.4f}, {v['ci95'][1]:.4f}]", "Significativa": v["significativa"]}
    for key, v in pairwise.items()
])
display(pairwise_df)

In [ ]:
# AUROC por clase en el test set
auroc_df = pd.DataFrame({name: eval_results[name]["auroc_per_class"] for name in EVAL_NAMES}, index=DISEASE_LABELS)
auroc_df["positivos_test"] = y_test.sum(axis=0).astype(int)
display(auroc_df.round(3))

# Grafica de puntos (no barras): el eje de AUROC no empieza en 0, y una barra truncada exageraria las diferencias
order = auroc_df[EVAL_NAMES].mean(axis=1).sort_values().index  # clases ordenadas por AUROC promedio (mejor arriba)
fig, ax = plt.subplots(figsize=(8, 8.5))
y = np.arange(len(order))
values = auroc_df.loc[order, EVAL_NAMES].to_numpy(dtype=float)
ax.hlines(y, np.nanmin(values, axis=1), np.nanmax(values, axis=1), color="#cccccc", linewidth=2, zorder=1)
for k, name in enumerate(EVAL_NAMES):
    ax.scatter(values[:, k], y, s=70, color=MODEL_COLORS[name], marker=MODEL_MARKERS[name], edgecolors="white", linewidths=1.2, zorder=2 + k, label=name)
ax.axvline(0.5, color="#666666", linewidth=1, linestyle="--", zorder=0)
ax.text(0.505, -0.6, "azar (0.5)", color="#444444", fontsize=8, va="center")
ax.set_yticks(y)
ax.set_yticklabels([f"{c} (n={auroc_df.loc[c, 'positivos_test']})" for c in order])
ax.set_ylim(-1, len(order))
ax.set_xlim(min(0.4, np.nanmin(values) - 0.05), 1.0)
ax.set_xlabel("AUROC (test)")
ax.set_title("AUROC por clase (n = positivos en test)")
style_axes(ax)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", color="#e5e5e5", linewidth=0.8)
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "auroc_per_class.png", dpi=150)
plt.show()

In [ ]:
# Costo computacional. El throughput de inferencia se mide solo en la GPU (batch sintetico ya en memoria),
# para que no lo domine la carga de imagenes desde disco, que es igual para todos los modelos.
def inference_throughput(model, n_iters=30, batch_size=BATCH_SIZE):
    model.to(device).eval()
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(5):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_iters):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - t0
    model.cpu()
    return n_iters * batch_size / elapsed


for name in MODEL_NAMES:
    training_results[name]["inference_imgs_per_sec"] = inference_throughput(trained_models[name])

cost_df = pd.DataFrame({
    "Parametros totales (M)": [training_results[n]["total_params"] / 1e6 for n in MODEL_NAMES],
    "Parametros entrenables (M)": [training_results[n]["trainable_params"] / 1e6 for n in MODEL_NAMES],
    "Tiempo/epoca (s)": [training_results[n]["avg_epoch_time_sec"] for n in MODEL_NAMES],
    "Memoria pico entrenamiento (MB)": [training_results[n]["peak_memory_mb"] for n in MODEL_NAMES],
    "Inferencia (img/s)": [training_results[n]["inference_imgs_per_sec"] for n in MODEL_NAMES],
}, index=MODEL_NAMES).round(2)
display(cost_df)

panels = [
    ("Parametros totales (M)", "Parametros totales (millones)", "{:.1f}M"),
    ("Parametros entrenables (M)", "Parametros entrenables (millones)", "{:.1f}M"),
    ("Tiempo/epoca (s)", "Tiempo por epoca (s)", "{:.1f}s"),
    ("Memoria pico entrenamiento (MB)", "Memoria GPU pico (MB)", "{:.0f}"),
    ("Inferencia (img/s)", "Inferencia (imagenes/s, mayor es mejor)", "{:.0f}"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 3.8))
for ax, (col, title, fmt) in zip(axes, panels):
    values = cost_df[col].to_numpy()
    bars = ax.bar(MODEL_NAMES, values, color=[MODEL_COLORS[n] for n in MODEL_NAMES], width=0.6)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), fmt.format(v),
                ha="center", va="bottom", fontsize=9, color="#222222")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, values.max() * 1.18)
    ax.tick_params(axis="x", labelsize=8)
    style_axes(ax)
fig.suptitle(f"Costo computacional (batch {BATCH_SIZE}, {IMG_SIZE}x{IMG_SIZE}, {str(device)})", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cost_comparison.png", dpi=150)
plt.show()

## 6. Interpretabilidad

Para varias imagenes del test set con un solo hallazgo, se visualiza donde "mira" cada modelo **para la clase real** de esa imagen:

- **Grad-CAM** (DenseNet121, ResNet18 y el hibrido MaxViT-T, que tambien produce mapas espaciales), sobre la ultima capa/etapa, con la clase real como objetivo.
- **ViT-B/16**: atencion del token `[CLS]` hacia los patches en el ultimo bloque, y **attention rollout** (Abnar & Zuidema, 2020), que combina la atencion de los 12 bloques (incluyendo las conexiones residuales) y suele ser mas fiel que mirar solo el ultimo bloque. Ambos mapas son independientes de la clase.

Son ejemplos cualitativos, no una medida de localizacion.

In [ ]:
# Grad-CAM para DenseNet121, ResNet18 y MaxViT-T. Si 'grad-cam' no esta instalado: pip install grad-cam
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
except ImportError:
    print("pytorch-grad-cam no instalado -- corre 'pip install grad-cam' para habilitar esta celda.")
    GRADCAM_AVAILABLE = False

# Ejemplos: imagenes del test con exactamente un hallazgo, cada una de una clase distinta (las mas frecuentes en test)
N_EXAMPLES = 3
single_finding = np.where(y_test.sum(axis=1) == 1)[0]
example_idx = []
for cls in np.argsort(-y_test.sum(axis=0)):
    candidates = [i for i in single_finding if y_test[i, cls] == 1]
    if candidates:
        example_idx.append(candidates[0])
    if len(example_idx) == N_EXAMPLES:
        break

def unnormalize(img_tensor):
    img = img_tensor.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1).astype(np.float32)

examples = []
for i in example_idx:
    img, label = test_dataset[i]
    cls = int(np.argmax(label.numpy()))
    examples.append({"tensor": img.unsqueeze(0), "img": unnormalize(img), "cls": cls})

gradcam_target_layers = {
    "densenet121": lambda m: [m.features.norm5],
    "resnet18": lambda m: [m.layer4[-1]],
    "maxvit_t": lambda m: [m.blocks[-1]],  # salida de la ultima etapa: mapa de 7x7
}

if GRADCAM_AVAILABLE:
    cols = ["original"] + list(gradcam_target_layers)
    fig, axes = plt.subplots(len(examples), len(cols), figsize=(4 * len(cols), 4 * len(examples)), squeeze=False)
    for r, ex in enumerate(examples):
        axes[r, 0].imshow(ex["img"])
        axes[r, 0].set_title(f"Real: {DISEASE_LABELS[ex['cls']]}", fontsize=10)
        for c, (name, layer_fn) in enumerate(gradcam_target_layers.items(), start=1):
            model = trained_models[name].to(device).eval()
            input_tensor = ex["tensor"].to(device)
            with torch.no_grad():
                prob = torch.sigmoid(model(input_tensor))[0, ex["cls"]].item()
            with GradCAM(model=model, target_layers=layer_fn(model)) as cam:
                grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(ex["cls"])])[0]
            model.cpu()
            axes[r, c].imshow(show_cam_on_image(ex["img"], grayscale_cam, use_rgb=True))
            axes[r, c].set_title(f"Grad-CAM {name}\np({DISEASE_LABELS[ex['cls']]}) = {prob:.2f}", fontsize=10)
        for ax in axes[r]:
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "gradcam_examples.png", dpi=120)
    plt.show()

In [ ]:
# Mapas de atencion para ViT-B/16. torchvision no expone los pesos de atencion en el forward normal, asi que
# se reproduce el forward de cada EncoderBlock (identico al de torchvision) pidiendo los pesos de atencion.

def vit_attention_maps(vit, x):
    """Regresa (atencion del CLS en el ultimo bloque, attention rollout), ambos como mapas 14x14 en [0, 1]."""
    with torch.no_grad():
        tokens = vit._process_input(x)
        tokens = torch.cat([vit.class_token.expand(tokens.shape[0], -1, -1), tokens], dim=1)
        h = vit.encoder.dropout(tokens + vit.encoder.pos_embedding)
        attentions = []
        for block in vit.encoder.layers:
            y = block.ln_1(h)
            y, attn = block.self_attention(y, y, y, need_weights=True, average_attn_weights=True)
            h = h + block.dropout(y)
            h = h + block.mlp(block.ln_2(h))
            attentions.append(attn[0])  # [seq_len, seq_len], promedio de las cabezas
        # Chequeo: el forward manual tiene que coincidir con el del modelo
        assert torch.allclose(vit.encoder.ln(h), vit.encoder(tokens), atol=1e-3), "El forward manual del ViT no coincide"

        # Rollout: A_l = 0.5 * atencion + 0.5 * identidad (conexion residual), normalizada por filas, y se multiplican
        identity = torch.eye(attentions[0].shape[0], device=x.device)
        rollout = identity
        for attn in attentions:
            a = 0.5 * attn + 0.5 * identity
            rollout = (a / a.sum(dim=-1, keepdim=True)) @ rollout

    def cls_to_map(row):
        v = row[1:].cpu().numpy()  # excluye el propio CLS
        g = int(np.sqrt(v.shape[0]))
        v = v.reshape(g, g)
        return (v - v.min()) / (v.max() - v.min() + 1e-8)

    return cls_to_map(attentions[-1][0]), cls_to_map(rollout[0])


vit_model = trained_models["vit_b_16"].to(device).eval()
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)), squeeze=False)
for r, ex in enumerate(examples):
    input_tensor = ex["tensor"].to(device)
    with torch.no_grad():
        prob = torch.sigmoid(vit_model(input_tensor))[0, ex["cls"]].item()
    last_map, rollout_map = vit_attention_maps(vit_model, input_tensor)
    h, w = ex["img"].shape[:2]
    extent = (-0.5, w - 0.5, h - 0.5, -0.5)  # el mapa 14x14 se estira exactamente sobre la radiografia
    axes[r, 0].imshow(ex["img"])
    axes[r, 0].set_title(f"Real: {DISEASE_LABELS[ex['cls']]}\np(ViT) = {prob:.2f}", fontsize=10)
    for c, (title, amap) in enumerate([("Atencion [CLS], ultimo bloque", last_map), ("Attention rollout (12 bloques)", rollout_map)], start=1):
        axes[r, c].imshow(ex["img"])
        axes[r, c].imshow(amap, cmap="jet", alpha=0.45, extent=extent, interpolation="bilinear")
        axes[r, c].set_title(title, fontsize=10)
    for ax in axes[r]:
        ax.axis("off")
vit_model.cpu()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "attention_example.png", dpi=120)
plt.show()

## 7. Exportar resultados

`results/metrics.json` queda como la fuente unica de verdad para el paper y las slides: no hace falta transcribir numeros a mano ni volver a entrenar para armar tablas.

In [ ]:
final_results = {
    "config": {
        "n_samples": N_SAMPLES,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "train_frac": TRAIN_FRAC,
        "val_frac": VAL_FRAC,
        "split": "por paciente (Patient ID)",
        "seed": SEED,
        "n_bootstrap": N_BOOTSTRAP,
        "device": str(device),
        "device_name": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
        "torch_version": torch.__version__,
        "full_finetuning": FULL_FINETUNING,
        "low_power": {"gpu_duty_sleep": GPU_DUTY_SLEEP, "cooldown_between_epochs_sec": COOLDOWN_BETWEEN_EPOCHS_SEC,
                      "cooldown_between_models_sec": COOLDOWN_BETWEEN_MODELS_SEC, "num_workers": NUM_WORKERS,
                      "nota": "epoch_times_sec excluye las pausas de bajo consumo"},
    },
    "split_sizes": {
        s: {"images": len(d), "patients": int(d[PATIENT_COL].nunique())}
        for s, d in [("train", df_train), ("val", df_val), ("test", df_test)]
    },
    "positives_per_class": prevalence_df[["train", "val", "test"]].to_dict(),
    "trivial_accuracy_all_negative": trivial_accuracy,
    "models": {},
    "pairwise_bootstrap": pairwise,
    "ensemble": {**eval_results["ensemble"], "members": MODEL_NAMES},
}

for name in MODEL_NAMES:
    final_results["models"][name] = {
        **eval_results[name],
        "total_params": training_results[name]["total_params"],
        "trainable_params": training_results[name]["trainable_params"],
        "best_epoch": training_results[name]["best_epoch"],
        "avg_epoch_time_sec": training_results[name]["avg_epoch_time_sec"],
        "epoch_times_sec": training_results[name]["history"]["epoch_time_sec"],
        "peak_memory_mb": training_results[name]["peak_memory_mb"],
        "inference_imgs_per_sec": training_results[name]["inference_imgs_per_sec"],
        "train_loss_history": training_results[name]["history"]["train_loss"],
        "val_loss_history": training_results[name]["history"]["val_loss"],
    }

def to_jsonable(o):
    """Convierte tipos de numpy a nativos y NaN a null (p. ej. AUROC de una clase sin positivos en test)."""
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if np.isnan(o) else float(o)
    return o


with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(final_results), f, indent=2, ensure_ascii=False)

print("Resultados guardados en", RESULTS_DIR / "metrics.json")
comparison_df.join(cost_df)

## 8. Limitaciones (para la discusion del paper)

*Escritas para la corrida principal de 12k; en esta corrida no aplica la del subconjunto, pero si las demas.*

- **Etiquetas ruidosas:** las etiquetas de ChestX-ray14 se extrajeron automaticamente de los reportes radiologicos con NLP (Wang et al. 2017 reportan ~90% de precision), asi que parte del error es ruido de etiqueta, no del modelo.
- **Subconjunto y presupuesto:** ~11% del dataset (12k imagenes) y 5 epocas con fine-tuning parcial (ultimo bloque + capa de salida). Los AUROC no son comparables directamente con CheXNet (dataset completo, fine-tuning completo); el objetivo es comparar arquitecturas bajo el mismo presupuesto.
- **Clases muy raras:** Hernia (y en menor medida Pneumonia, Edema, Fibrosis) tiene muy pocos positivos en test; sus metricas por clase son inestables y pesan igual que las demas en los promedios macro.
- **Resolucion:** las radiografias (1024x1024) se reducen a 224x224; hallazgos pequenos como nodulos pierden detalle.
- **Mismo protocolo, distinta capacidad ajustable:** se descongela el ultimo bloque de cada modelo, lo que da 2.2M / 8.4M / 7.1M / 9.2M parametros entrenables (DenseNet121 / ResNet18 / ViT-B/16 / MaxViT-T); parte de las diferencias viene de ahi: ResNet18 (75% de la red ajustada) se sobreajusta, y MaxViT-T no llego a converger en 5 epocas (su pérdida de validacion seguia bajando).
- **Una sola semilla:** los intervalos de confianza por bootstrap capturan la variabilidad del test set, no la del entrenamiento (inicializacion del head, orden de los batches). Una repeticion independiente (`results/subset_12k/`) cambio el orden de los modelos individuales con variaciones de hasta 0.010 de AUROC, del mismo orden que las diferencias entre arquitecturas.
- **Hardware:** AMD Radeon RX 6800 XT con PyTorch + ROCm nightly (sin soporte oficial). En ROCm la atencion "memory efficient" no esta disponible, asi que el tiempo y la memoria de los modelos con atencion (ViT-B/16, MaxViT-T) son pesimistas frente a una GPU NVIDIA; los tiempos absolutos no son comparables con los de la literatura.
- **Tiempo por epoca limitado por la carga de datos:** decodificar y redimensionar PNGs de 1024x1024 (4 workers) domina el tiempo por epoca de las CNNs, por eso DenseNet121 y ResNet18 salen parecidos aunque en computo puro ResNet18 es ~4x mas rapido. Para comparar el costo de computo de los modelos, la metrica limpia es el throughput de inferencia (medido solo en GPU); pre-redimensionar las imagenes a 256px quitaria este cuello de botella.
- **Interpretabilidad cualitativa:** pocos ejemplos; Grad-CAM y los mapas de atencion no son una medida de localizacion (el dataset incluye bounding boxes para ~1k imagenes en `BBox_List_2017.csv`, que permitirian evaluarla cuantitativamente como trabajo futuro).